# 3.7 Weight Decay

- Overfitting
    - One solution to overfitting
    - Alternative to collecting more training data (collection is costly, time consuming, or out of our control)

For now, assume we already have as much high-quality data as we can in the given dataset. Recall the polynomial regression example from 3.6.2:

- *Given training data consisting of a singular feature $x$ and a corresponding real-valued label $y$, we try to find the polynomial of degree $d$ such that
$\hat{y}=\sum_{i=0}^{d}x^iw_i$ (polynomial theorem) for estimating the label $y$. This is just a linear regression problem where our features are given by the powers of $x$, the model's weights are given by $w_i$, and the bias is given by $w_0$ since $\forall x \in \mathbb{R} \ x^0=1$. As such we can just use the squared error as our loss function.*

- We can limit model capacity by tweaking polynomial degree.
- Limiting # of features is a popular method for mitigating overfitting
    - However, simply removing features is too blunt.

Why not just remove features? Consider high-dimension input:
- *Monomials* - products of powers of variables (e.g. $x_1^2x_2$, $x_3x_5^2$)
- Given $k$ variables, number of monomials of degree $d$ is ${k-1+d}\choose{k-1}$ (number of terms with degree $d$ grows rapidly as $d$ increases)
    - This means even changes from degree 2 to degree 3 dramatically incrase model complexity.

Thus, we will investigate a more fine-grained tool for adjusting function complexity.

In [1]:
%matplotlib inline
import torch
from torch import nn
from d2l import torch as d2l

## 3.7.1: Norms and Weight Decay

Instead of directly manipulating parameters, weight decay restricts values parameters can take. 
    - $l_2$ regularization is specifically weight decay optimized by minibatch stochastic gradient descent
    - motivatd by basic intuition that among all functions $f$, $f=0$ (assigning 0 to all inputs) is the simplest, and we can measure complexity of a function by the distance of its parameters from zero

One simple interpretation: measure complexity of a linear function $f(\bold{x})=\bold{w}^T\bold{x}$ by some norm of the weight vector (e.g. $||\bold{w}||^2$)

- Most common method for ensuring small weight vector is to add its norm as a penalty term to the problem of minimizng loss using the $l_p$ norm
- Thus we replace our original objective of **minimizing prediction loss on training labels** with **minimizing sum of prediction loss and penalty terms**

Note that if weight vector grows too large, our learning algorithm may focus on minimizing $|\bold{w}|^2$ rather than minimizing training error. 

#### Illustration: example from 3.1, loss for linear regression

$$
\displaystyle
L(\bold{w},b) = \frac{1}{n}\sum^n_{i=1}\frac{1}{2}(\bold{w}^T\bold{x}^i+b-y^(i))
$$

AKA mean squared error (MSE) over all observations, where $\bold{x}^i$ are the features, $y^(i)$ is the label for any data example $i$, and $(\bold{w},b)$ are the weights and bias parameters, respectively. 

- To penalize size of weight vecctor, we must somehow add $||\bold{w}||^2$ to the loss function.
- How should model trace off standard loss for new additive penalty?
    - Ans: *regularization constant $\lambda$*, nonnegative hyperparameter used to fit to validation data.
$$
\displaystyle
L(\bold{w},b)+\frac{\lambda}{2}||\bold{w}||^2
$$
- $\lambda=0 \ \rightarrow \ Loss = L(\bold{w},b)$
-$\lambda \gt 0 \ \rightarrow $ we restrict the size of $||\bold{w}||$
- The $\frac{1}{2}$ applied to $\lambda$ is simply to make applying derivatives 
- Why use $l_2$ norm?
    - Convenience
        - Squaring $l_2$ norm gives sum of squares of each elem in $\bold{w}$
        - Computing derivatives of this is simply applying sum rule to sum of squares
            * "Sum of derivatives == derivative of sum"
    - Places outsize penalty on large components of weight vector, biasing learning algorithms towards models that distribute weight evenly across a larger number of features
        - To visualize: imagine one weight of $\bold{w}$ blowing the others out by 1000x - $1000^2=1,000,000$, resulting in a humongous $l_2$ norm and thus a very large loss
- What effect does $l_1$ norm have?
    - $l_1$ penalites lead to models that concentrate weights on small sets of features by clearing other weights to 0.
        - Recap: $l_1$ is sum of all components, so a $l_1$ penalty is like using a credit system for weights with a soft 'limit'. Model must choose best allocation of limited weight credits to minimize loss
        - $\bold{w}_i=1000$ example: a model with $l_1$ norm will do __nothing__ (or next to nothing) as long as output predictions are accurate!
    - Effective method for feature selection, which may be desireable for other reasons
        - E.g. model relies only on a few features -> only collect those features, drop all others

- $l_2$-regularized linear regression models == "ridge regression algorithm"
- $l_1$-regularized linear regression models == "lasso regression"

Minibatch stochastic gradient descent algorithm:
$$
\displaystyle
\bold{w} \leftarrow (1 - \eta \lambda) \bold{w} - \frac{\eta}{|\Beta|} \sum_{i \in \Beta} (\bold{x}^{(i)}) \cdot (\bold{w}^T\bold{x}^{(i)}+b-y^{(i)})
$$
- $\eta$ (\eta): Learning rate
- $\lambda$ (\lambda): Hyperparameter to manage tradeoff between Loss objective and $l_i$ penalty objective
- $\Beta$ (\Beta): set of data 
- $(1 - \eta \lambda) \bold{w}$: 
    * We start by distributing $\eta$ over both the loss term and regularization term 
    * Now considering only the weight and regularization part of the weight update, we have $\bold{w}-\eta\nabla\frac{\lambda}{2}||\bold{w}||^2 = \bold{w}-\eta\lambda||\bold{w}||$
    * This is the first two set of terms
- $(\bold{x}^{(i)}) \cdot (\bold{w}^T\bold{x}^{(i)}+b-y^{(i)})$
    - $\bold{w}^T\bold{x}^(i)+b-y^(i)$: Original residual, or $e^{(i)}$
    - $\frac{1}{2}e^{(i)^2}$ is 1/2 MSE
    - $\frac{\partial}{\partial \bold{w}}\frac{1}{2}e^{(i)^2} = e^(i) \cdot \bold{x}^(i)$, where $e^(i)$ is a scalar value.
    - Thus we can average loss over the entire minibatch as $\frac{\eta}{|\Beta|} \sum_{i \in \Beta} (\bold{x}^{(i)}) \cdot (\bold{w}^T\bold{x}^{(i)}+b-y^{(i)})$

- $\frac{\eta}{|\Beta|}$: Divide total loss over all samples of $||\Beta||$. Since we distribute $\eta$ earlier to simplify the regularization term, we also include the multipler $\eta$ here.

Minibatch SGD vs SGD: Minibatch backpropagates loss after training on a batch of examples, SGD backpropagates after every point.

GD backpropagates once on a whole dataset

## 3.7.2: High-Dimensional Linear Regression